# Grupo 4 — Clasificación de sentimientos en reseñas Amazon

Práctica reproducible: dataset de bittlingmayer, dos clases, principalmente inglés.
Este notebook ejecuta el código del repositorio y muestra resultados calculados.
No contiene resultados precargados ni atribuye código propio a otros autores.

Ejecuta las celdas en orden. En Colab se descargan ~517 MB; no se necesita GPU. En Jupyter abre el notebook desde el repositorio clonado.

In [ ]:
from pathlib import Path
import os, subprocess, sys
if "google.colab" in sys.modules:
    root = Path("/content/clasificador")
    if not root.exists():
        subprocess.run(["git", "clone", "https://github.com/thejhonmol/clasificador.git", str(root)], check=True)
else:
    root = Path.cwd()
    if root.name == "notebooks":
        root = root.parent
os.chdir(root)
assert (root / "requirements.txt").exists(), "Abre el notebook desde el repositorio."
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)

## 4.1 Dataset y etiquetado

Se leen los archivos oficiales train/test por separado. Etiqueta 1 → negativo (0); etiqueta 2 → positivo (1). No hay neutral. Si ya descargaste los archivos, cambia DATA. Kaggle puede solicitar una descarga manual con tu sesión; no pegues credenciales aquí.

In [ ]:
DATA = root / "data"
if not all((DATA / name).exists() for name in ["train.ft.txt.bz2", "test.ft.txt.bz2"]):
    subprocess.run([sys.executable, "-m", "clasificador.download", "--output", str(DATA)], check=True)

## 4.2 Limpieza y representación

Se conserva la negación, se normalizan HTML/URLs/espacios y se aplica TF-IDF de unigramas y bigramas. No se ajusta vocabulario con test. Este texto de ejemplo es sintético.

In [ ]:
from clasificador.data import clean_text
clean_text("<p>This is NOT a good product &amp; it does not work.</p>")

## 4.3 Partición y 4.4 Modelos

Muestreo uniforme de 20 000 filas de train y 5 000 de test; tamaños efectivos pueden reducirse por duplicados. Train se separa en ajuste (80 %) y validación (20 %). GridSearchCV de tres folds ajusta los hiperparámetros de NB, regresión logística y SVM. La familia se elige por F1 macro de validación antes de evaluar test.

El lector recorre el corpus completo para muestrear. Espera a que termine; los archivos se guardan en una carpeta única.

In [ ]:
from datetime import datetime
OUT = root / "outputs" / datetime.now().strftime("notebook_%Y%m%d_%H%M%S_%f")
subprocess.run([sys.executable, "-m", "clasificador.train",
    "--train", str(DATA / "train.ft.txt.bz2"), "--test", str(DATA / "test.ft.txt.bz2"),
    "--output", str(OUT), "--train-size", "20000", "--test-size", "5000", "--seed", "42"], check=True)

## 4.5 Explicación del código

`data.py` lee y limpia; `train.py` construye Pipeline, busca parámetros, selecciona y exporta; `predict.py` reutiliza la limpieza y el pipeline guardado. Consulta `docs/METODOLOGIA.md` para las decisiones y límites.

In [ ]:
import json
manifest = json.loads((OUT / "manifest.json").read_text())
print("Tamaños:", manifest["sizes"])
print("Ganador por validación:", manifest["selected_model"])
print("Filas del corpus:", manifest["train"]["total_rows"], manifest["test"]["total_rows"])

## 5. Resultados y discusión

Estas tablas corresponden a la ejecución anterior. No cambies el modelo ganador usando test. Revisa el soporte por clase, los errores y las limitaciones antes de escribir conclusiones.

In [ ]:
import pandas as pd
from IPython.display import display, Image, Markdown
display(pd.read_csv(OUT / "metrics.csv"))
display(Markdown((OUT / "RESULTADOS.md").read_text()))
for name in ["naive_bayes", "logistic_regression", "linear_svm"]:
    display(Image(filename=str(OUT / f"confusion_{name}.png")))

In [ ]:
display(pd.read_csv(OUT / "errors.csv").head(10))
print("Los IDs source_line se refieren a líneas base 1 del archivo test original.")

## Predicción interactiva

Usa una reseña en inglés. La salida es binaria, no una probabilidad ni una validación en español.

In [ ]:
review = "This product works perfectly and I really like it"
subprocess.run([sys.executable, "-m", "clasificador.predict", "--model", str(OUT / "model.joblib"), "--text", review], check=True)

## Fuentes

- [Dataset de Kaggle](https://www.kaggle.com/datasets/bittlingmayer/amazonreviews)
- [TF-IDF](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html)
- [GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html)
- `docs/REFERENCIAS.md` y `docs/METODOLOGIA.md` en este repositorio.

Las conclusiones académicas deben basarse en la ejecución observada. No se han escrito conclusiones ficticias.